# 🚀 NostalgiaForInfinityX8 - Google Colab High-Speed Backtesting Suite
### Tối ưu hóa Backtest đa luồng tốc độ cao với Google Colab Cloud + Đồng bộ Google Drive

**Tính năng:**
1. ⚡ **Tốc độ siêu nhanh**: Chạy trên hạ tầng Linux Cloud của Google Colab (CPU đa nhân, RAM 13GB+).
2. 💾 **Đồng bộ Google Drive tự động**: Dữ liệu nến (`feather`) và kết quả backtest (`.zip`) được lưu vĩnh viễn trên Google Drive.
3. 🛡️ **Tích hợp đầy đủ chiến lược v3.6.1 Shield Triad**: Đồng bộ từ repo [`tminhnhat/NostalgiaForInfinity`](https://github.com/tminhnhat/NostalgiaForInfinity).
4. 🏎️ **I/O Cache Optimizer**: Sao chép dữ liệu từ Google Drive sang ổ đĩa SSD NVMe cục bộ của Colab để triệt tiêu độ trễ mạng.
5. 📊 **Báo cáo chuyên sâu tự động**: Phân tích trực quan Winrate, Drawdown, Calmar, Lợi nhuận Short/Long và chi tiết các Tag.

## 📌 Bước 1: Kết nối Google Drive
Chạy ô này để cấp quyền liên kết với Google Drive cá nhân của bạn. Dữ liệu nến và kết quả backtest sẽ được lưu tự động tại thư mục `botx8` trên Google Drive.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive vào môi trường Colab
drive.mount('/content/drive')

# Thiết lập các thư mục lưu trữ vĩnh viễn trên Google Drive
DRIVE_BASE_DIR = '/content/drive/MyDrive/botx8'
DRIVE_DATA_DIR = os.path.join(DRIVE_BASE_DIR, 'data/binance')
DRIVE_RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, 'backtest_results')

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

print('✅ Đã kết nối Google Drive thành công!')
print(f'📁 Thư mục dữ liệu: {DRIVE_DATA_DIR}')
print(f'📁 Thư mục kết quả: {DRIVE_RESULTS_DIR}')

## 📌 Bước 2: Cài đặt TA-Lib C Library & Freqtrade
Cài đặt thư viện TA-Lib chuẩn C và Freqtrade phiên bản tối ưu trên nền tảng Linux.

In [ ]:
# Cài đặt thư viện TA-Lib C
!echo '⏳ Đang cài đặt TA-Lib C Library...'
!wget -q http://prdownloads.sourceforge.net/ta-lib/ta-lib-0.4.0-src.tar.gz
!tar -xzf ta-lib-0.4.0-src.tar.gz
%cd ta-lib/
!./configure --prefix=/usr > /dev/null
!make -j$(nproc) > /dev/null
!make install > /dev/null
%cd ..
!rm -rf ta-lib ta-lib-0.4.0-src.tar.gz

# Cài đặt TA-Lib Python wrapper & Freqtrade
!echo '⏳ Đang cài đặt Freqtrade và Python dependencies...'
!pip install -q TA-Lib
!pip install -q freqtrade pyarrow fastparquet numba

!echo '✅ Cài đặt môi trường hoàn tất!'
!freqtrade --version

## 📌 Bước 3: Đồng bộ Repository NostalgiaForInfinity
Kéo toàn bộ mã nguồn chiến lược mới nhất (`v3.6.1 Shield Triad`) từ GitHub repo.

In [ ]:
REPO_DIR = '/content/botx8'
REPO_URL = 'https://github.com/tminhnhat/NostalgiaForInfinity.git'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main
    !git reset --hard origin/main

%cd {REPO_DIR}

# Đảm bảo đồng bộ nhị phân giữa root và user_data/strategies
!cp -f NostalgiaForInfinityX8_Combined.py user_data/strategies/
!cp -f NostalgiaForInfinityX8_ShortSignals.py user_data/strategies/

print('✅ Repository đã sẵn sàng tại:', REPO_DIR)
!git log -n 1 --oneline

## 📌 Bước 4: Tối ưu hóa I/O Dữ liệu (Google Drive ➡️ NVMe SSD Cục bộ)
Để đạt tốc độ đọc nến tối đa, dữ liệu từ Google Drive được sao chép vào bộ nhớ đệm SSD NVMe của Colab.
*(Nếu trên Google Drive chưa có dữ liệu, bạn có thể chạy ô tải dữ liệu bên dưới)*.

In [ ]:
import shutil

LOCAL_DATA_DIR = os.path.join(REPO_DIR, 'user_data/data/binance')
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Kiem tra du lieu nen (.zip) hoac thu muc nen feather tren Google Drive
DRIVE_ZIP = os.path.join(DRIVE_BASE_DIR, 'data_binance.zip')
drive_files = []
if os.path.exists(DRIVE_DATA_DIR):
    drive_files = [f for f in os.listdir(DRIVE_DATA_DIR) if f.endswith('.feather') or os.path.isdir(os.path.join(DRIVE_DATA_DIR, f))]

if os.path.exists(DRIVE_ZIP):
    print(f'⚡ Tim thay goi du lieu nen sieu toc: {DRIVE_ZIP}')
    print('🚀 Dang giai nen truc tiep vao NVMe SSD Colab...')
    !unzip -q -o "{DRIVE_ZIP}" -d "{REPO_DIR}/user_data/"
    print('✅ Giai nen du lieu hoan tat trong vai giay!')
elif drive_files:
    print(f'⚡ Tim thay du lieu tren Google Drive ({len(drive_files)} muc). Dang dong bo vao NVMe SSD Colab...')
    !cp -ru "{DRIVE_DATA_DIR}"/* "{LOCAL_DATA_DIR}/"
    print('✅ Dong bo du lieu tu Google Drive hoan tat!')
else:
    print('ℹ️ Chua co du lieu tren Google Drive.')
    print('👉 Neu can tai moi tu san Binance Futures, hay chay o Tai Du lieu ben duoi.')

### *(Tùy chọn)* Tải dữ liệu nến Binance Futures nếu Google Drive chưa có
Chạy ô này nếu bạn cần tải nến mới. Sau khi tải, dữ liệu sẽ tự động được lưu vĩnh viễn lên Google Drive để các lần sau không cần tải lại.

In [ ]:
# Tải dữ liệu nến 5m và 1h cho các cặp giao dịch trong cấu hình
# (Bỏ comment dòng bên dưới nếu muốn tải dữ liệu mới)
# !freqtrade download-data --config configs/backtest_config.json --trading-mode futures --timeframes 5m 15m 1h 4h 1d --timerange 20260416-20261014

# Đồng bộ ngược dữ liệu vừa tải lên Google Drive để lưu trữ vĩnh viễn
# !cp -ru "{LOCAL_DATA_DIR}"/* "{DRIVE_DATA_DIR}/"
# print('✅ Đã sao lưu dữ liệu nến vào Google Drive!')

## 📌 Bước 5: Kích hoạt Bản vá Vượt Chặn IP Binance (Offline Market Loader) & Tối ưu RAM
Bản vá này giúp Freqtrade nạp toàn bộ cấu trúc thị trường từ cache offline 100%, triệt tiêu hoàn toàn lỗi **HTTP 451 (Binance Restricted Location)** do hạ tầng Colab đặt tại Mỹ.

In [ ]:
import os, gc, psutil

# 1. Kiểm tra dung lượng RAM hệ thống Colab Cloud
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
ram_avail = psutil.virtual_memory().available / (1024 ** 3)
gc.collect()
print(f'✅ Google Colab Cloud RAM: {ram_gb:.1f} GB tổng (Khả dụng: {ram_avail:.1f} GB)')

# 2. Kích hoạt bản vá Offline Market Loader (Vượt triệt để lỗi HTTP 451 do Binance chặn IP Colab / US)
import re
import freqtrade.exchange.exchange as fe
fe_file = fe.__file__
with open(fe_file, 'r', encoding='utf-8') as f:
    fe_code = f.read()

correct_patch = '''
        # --- OFFLINE MARKETS PATCH (Bypass Binance US IP block 451) ---
        import os, pickle, gzip
        for _p in [
            os.path.join(self._config.get('user_data_dir', 'user_data'), 'markets_binance_futures.pkl.gz'),
            'user_data/markets_binance_futures.pkl.gz',
            '/content/botx8/user_data/markets_binance_futures.pkl.gz',
            '/content/drive/MyDrive/botx8/markets_binance_futures.pkl.gz'
        ]:
            if os.path.exists(_p):
                with gzip.open(_p, 'rb') as _f:
                    _m = pickle.load(_f)
                self._api_async.markets = _m
                self._api_async.set_markets(_m)
                return None
        # -------------------------------------------------------------
'''

if 'OFFLINE MARKETS PATCH' in fe_code:
    fe_code = re.sub(r'\n\s*# --- OFFLINE MARKETS PATCH[\s\S]*?# -{20,}\s*\n', correct_patch, fe_code)
    with open(fe_file, 'w', encoding='utf-8') as f:
        f.write(fe_code)
    print("✅ Đã cập nhật lại bản vá Offline Market Loader đầy đủ 'import os'!")
else:
    target = '    def _load_async_markets(self, reload: bool = False) -> None:\n        try:'
    rep = '    def _load_async_markets(self, reload: bool = False) -> None:' + correct_patch + '        try:'
    fe_code = fe_code.replace(target, rep)
    with open(fe_file, 'w', encoding='utf-8') as f:
        f.write(fe_code)
    print('✅ Đã kích hoạt bản vá Offline Market Loader thành công!')


## 📌 Bước 6: Khởi chạy Backtest Tốc độ cao 🚀
Chạy Backtest toàn diện từ **15/04/2026 đến 18/10/2026** cho danh mục 101 cặp coin trong cấu hình `pairlist-backtest-static-binance-futures-usdt.json` với chiến lược `NostalgiaForInfinityX8_Combined` (v18.0.143 + v3.6.1 Shield Triad).

In [ ]:
import os, re

# 1. Tự động xác định và đồng bộ mã nguồn mới nhất từ GitHub
REPO_DIR = '/content/botx8'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/tminhnhat/NostalgiaForInfinity.git {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin main && git -C {REPO_DIR} reset --hard origin/main

%cd {REPO_DIR}

# 2. Đảm bảo file offline markets cache sẵn sàng (vượt triệt để lỗi 451)
MARKETS_CACHE = os.path.join(REPO_DIR, 'user_data/markets_binance_futures.pkl.gz')
if not os.path.exists(MARKETS_CACHE):
    DRIVE_CACHE = '/content/drive/MyDrive/botx8/markets_binance_futures.pkl.gz'
    if os.path.exists(DRIVE_CACHE):
        !cp -f "{DRIVE_CACHE}" "{MARKETS_CACHE}"
        print('✅ Đã đồng bộ file offline markets cache từ Google Drive!')

# 3. Kích hoạt và cập nhật bản vá Offline Markets cho Freqtrade (đầy đủ import os, pickle, gzip)
import freqtrade.exchange.exchange as fe
fe_file = fe.__file__
with open(fe_file, 'r', encoding='utf-8') as f:
    fe_code = f.read()

correct_patch = '''
        # --- OFFLINE MARKETS PATCH (Bypass Binance US IP block 451) ---
        import os, pickle, gzip
        for _p in [
            os.path.join(self._config.get('user_data_dir', 'user_data'), 'markets_binance_futures.pkl.gz'),
            'user_data/markets_binance_futures.pkl.gz',
            '/content/botx8/user_data/markets_binance_futures.pkl.gz',
            '/content/drive/MyDrive/botx8/markets_binance_futures.pkl.gz'
        ]:
            if os.path.exists(_p):
                with gzip.open(_p, 'rb') as _f:
                    _m = pickle.load(_f)
                self._api_async.markets = _m
                self._api_async.set_markets(_m)
                return None
        # -------------------------------------------------------------
'''

if 'OFFLINE MARKETS PATCH' in fe_code:
    fe_code = re.sub(r'\n\s*# --- OFFLINE MARKETS PATCH[\s\S]*?# -{20,}\s*\n', correct_patch, fe_code)
    with open(fe_file, 'w', encoding='utf-8') as f:
        f.write(fe_code)
    print("✅ Đã cập nhật bản vá Offline Market Loader đầy đủ 'import os'!")
else:
    target = '    def _load_async_markets(self, reload: bool = False) -> None:\n        try:'
    rep = '    def _load_async_markets(self, reload: bool = False) -> None:' + correct_patch + '        try:'
    fe_code = fe_code.replace(target, rep)
    with open(fe_file, 'w', encoding='utf-8') as f:
        f.write(fe_code)
    print('✅ Đã kích hoạt bản vá Offline Market Loader: Vượt qua lỗi chặn IP Binance (HTTP 451) thành công!')

# 4. Kiểm tra dữ liệu nến trên ổ đĩa SSD NVMe
LOCAL_FUTURES_DATA = os.path.join(REPO_DIR, 'user_data/data/binance/futures')
if not os.path.exists(LOCAL_FUTURES_DATA) or len(os.listdir(LOCAL_FUTURES_DATA)) == 0:
    print('⚠️ Đang tự động giải nén nến từ Google Drive vào SSD Colab...')
    DRIVE_ZIP = '/content/drive/MyDrive/botx8/data_binance.zip'
    if os.path.exists(DRIVE_ZIP):
        !unzip -q -o "{DRIVE_ZIP}" -d "{REPO_DIR}/user_data/"
        print('✅ Đã giải nén dữ liệu nến thành công!')
    elif os.path.exists('/content/drive/MyDrive/botx8/data/binance'):
        !mkdir -p "{REPO_DIR}/user_data/data/binance"
        !cp -ru /content/drive/MyDrive/botx8/data/binance/* "{REPO_DIR}/user_data/data/binance/"
        print('✅ Đã đồng bộ dữ liệu nến từ Google Drive!')

# 5. Đồng bộ mã chiến lược mới nhất
!cp -f NostalgiaForInfinityX8_Combined.py user_data/strategies/
!cp -f NostalgiaForInfinityX8_ShortSignals.py user_data/strategies/

# 6. Khởi chạy Backtest
print('🚀 Bắt đầu chạy Backtest với Freqtrade...')
!freqtrade backtesting \
  --config configs/backtest_config.json \
  --config configs/pairlist-backtest-static-binance-futures-usdt.json \
  --strategy NostalgiaForInfinityX8_Combined \
  --strategy-path user_data/strategies/ \
  --timerange 20260415-20261019 \
  --breakdown day


## 📌 Bước 7: Tự động lưu Kết quả vào Google Drive & Xuất Báo Cáo
Sao lưu file `.zip` kết quả vào Google Drive vĩnh viễn và trích xuất bảng phân tích chỉ số chi tiết.

In [ ]:
import glob, shutil, json, zipfile, os
import pandas as pd

REPO_DIR = '/content/botx8'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/botx8/backtest_results'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

# 1. Tìm file kết quả backtest mới nhất (.zip hoặc .json)
local_results = sorted(glob.glob(os.path.join(REPO_DIR, 'user_data/backtest_results/*.zip')), key=os.path.getmtime)
if not local_results:
    local_results = sorted(glob.glob(os.path.join(REPO_DIR, 'user_data/backtest_results/*.json')), key=os.path.getmtime)
    local_results = [f for f in local_results if not f.endswith('.meta.json')]

# 2. Nếu trong thư mục tạm Colab chưa có, kiểm tra trên Google Drive
if not local_results:
    drive_results = sorted(glob.glob(os.path.join(DRIVE_RESULTS_DIR, '*.zip')), key=os.path.getmtime)
    if drive_results:
        print(f'ℹ️ Đã tìm thấy kết quả đã lưu trước đó trên Google Drive: {os.path.basename(drive_results[-1])}')
        local_results = drive_results

if not local_results:
    print('❌ Không tìm thấy file kết quả backtest trong user_data/backtest_results/!')
    print('👉 Chú ý: Bước 6 cần chạy hoàn tất (thường mất 5-8 phút cho 101 coin trong 180 ngày).')
    print('   Hãy kiểm tra ô Bước 6 xem vòng tròn đã chuyển thành dấu tích xanh [✓] chưa, hoặc có báo lỗi đỏ nào không.')
else:
    latest_result = local_results[-1]
    filename = os.path.basename(latest_result)
    dest_path = os.path.join(DRIVE_RESULTS_DIR, filename)
    if latest_result != dest_path:
        shutil.copy2(latest_result, dest_path)
        meta_file = latest_result.replace('.zip', '.meta.json')
        if os.path.exists(meta_file):
            shutil.copy2(meta_file, os.path.join(DRIVE_RESULTS_DIR, os.path.basename(meta_file)))
        print(f'✅ Đã lưu kết quả vĩnh viễn vào Google Drive: {dest_path}')
    
    # Đọc và hiển thị số liệu phân tích
    strat = None
    strat_name = 'NostalgiaForInfinityX8_Combined'
    if latest_result.endswith('.zip'):
        with zipfile.ZipFile(latest_result, 'r') as z:
            for name in z.namelist():
                if name.endswith('.json') and not name.endswith('.meta.json'):
                    data = json.loads(z.read(name))
                    strat_name = list(data['strategy'].keys())[0]
                    strat = data['strategy'][strat_name]
                    trades = pd.DataFrame(strat['trades'])
                    break
    elif latest_result.endswith('.json'):
        with open(latest_result, 'r', encoding='utf-8') as f:
            data = json.load(f)
            strat_name = list(data['strategy'].keys())[0]
            strat = data['strategy'][strat_name]
            trades = pd.DataFrame(strat['trades'])
    
    if strat is not None:
        shorts = trades[trades.get('is_short', False) == True] if 'is_short' in trades.columns else pd.DataFrame()
        longs = trades[trades.get('is_short', False) == False] if 'is_short' in trades.columns else trades
        
        print('\n' + '='*75)
        print(f"📊 KẾT QUẢ BACKTEST TOÀN DIỆN - {strat_name}")
        print('='*75)
        print(f"💰 Tổng Lợi Nhuận:        {strat['profit_total_abs']:.2f} USDT ({strat['profit_total']*100:.2f}%)")
        print(f"🎯 Tỷ Lệ Thắng (Winrate): {strat['winrate']*100:.2f}% ({strat['wins']} Thắng / {strat['losses']} Thua)")
        print(f"📉 Max Drawdown:          {strat['max_drawdown_account']*100:.2f}% ({strat['max_drawdown_abs']:.2f} USDT)")
        print(f"💎 Calmar Ratio:          {strat.get('calmar', 0):.2f}")
        print(f"⚖️ Profit Factor:         {strat.get('profit_factor', 0):.2f}")
        print('-'*75)
        if len(longs) > 0:
            print(f"🟢 Lợi Nhuận Long:        {longs['profit_abs'].sum():.2f} USDT | {len(longs)} lệnh (Winrate: {(longs['profit_abs']>0).sum()/len(longs)*100:.1f}%)")
        if len(shorts) > 0:
            print(f"🔴 Lợi Nhuận Short:       {shorts['profit_abs'].sum():.2f} USDT | {len(shorts)} lệnh (Winrate: {(shorts['profit_abs']>0).sum()/len(shorts)*100:.1f}%)")
        print('='*75)
